# PPO agents

> PPO based agent

In [ ]:
#| default_exp agents.rl.RL2ppo

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

# set logging level to INFO
logging.basicConfig(level=logging.INFO)

from abc import ABC, abstractmethod
from typing import Union, Optional, List, Tuple
import numpy as np
import os

from ddopai.envs.base import BaseEnvironment
from ddopai.agents.rl.mushroom_rl import MushroomBaseAgent
from ddopai.utils import MDPInfo, Parameter
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.RL_approximators import MLPState, MLPActor, RL2RNNActor, RL2RNNValue
from ddopai.envs.actionprocessors import ClipAction

from ddopai.dataloaders.base import BaseDataLoader
from mushroom_rl.policy import GaussianTorchPolicy
from mushroom_rl.policy import TorchPolicy
from mushroom_rl.core import Agent
from mushroom_rl.approximators import Regressor
#from mushroom_rl.approximators.parametric import TorchApproximator
from ddopai.RL_approximators import TorchApproximator
from mushroom_rl.utils.torch import to_float_tensor, update_optimizer_parameters
from mushroom_rl.utils.minibatches import minibatch_generator
from mushroom_rl.utils.dataset import parse_dataset, compute_J
from mushroom_rl.utils.value_functions import compute_gae
from mushroom_rl.utils.parameters import to_parameter

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
from itertools import chain
import time

In [ ]:

#| export

class GaussianTorchPolicyRL2(TorchPolicy):
    """
    Torch policy implementing a Gaussian policy with trainable standard
    deviation. The standard deviation is not state-dependent.
    """
    def __init__(self, network, input_shape, output_shape, mdp_info, std_0=1.,
                 use_cuda=False, **params):
        """
        Constructor.

        Args:
            network (object): the network class used to implement the mean regressor.
            input_shape (tuple): the shape of the state space.
            output_shape (tuple): the shape of the action space.
            env_info (MDPInfo): the environment information.
            std_0 (float, 1.): initial standard deviation.
            params (dict): parameters used by the network constructor.
        """
        super().__init__(use_cuda)

        self._action_dim = output_shape[0]

        self._mu = Regressor(TorchApproximator, input_shape, output_shape,
                             network=network, use_cuda=use_cuda, **params)
        self._predict_params = dict()
        max_a = mdp_info.action_space.high
        min_a = mdp_info.action_space.low
        self._delta_a = to_float_tensor(0.5 * (max_a - min_a), use_cuda=use_cuda)
        self._central_a = to_float_tensor(0.5 * (max_a + min_a), use_cuda=use_cuda)
        
        log_sigma_init = (torch.ones(self._action_dim) * np.log(std_0)).float()
        if self._use_cuda:
            log_sigma_init = log_sigma_init.cuda()

        self._log_sigma = nn.Parameter(log_sigma_init)

        # This attribute will store the distribution computed at each time step.
        self._last_dist = None

        self._add_save_attr(
            _action_dim='primitive',
            _mu='mushroom',
            _predict_params='pickle',
            _log_sigma='torch'
        )

    def draw_action_t(self, state, hidden_state):
        """
        Computes the action given a state and hidden state.
        
        Args:
            state (tensor): the current state.
            hidden_state (tensor or tuple): the hidden state for the RNN.
        
        Returns:
            action (tensor): sampled action.
            new_hidden_state: updated hidden state returned by the regressor.
        """
        dist, new_hidden_state = self.distribution_t(state, hidden_state)
        a_raw = dist.rsample()
        
        #a_than = torch.tanh(a_raw)
        #a_true = a_than * self._delta_a + self._central_a
        #self._last_a_raw = a_raw
        #self._last_transformed_a = a_true
        
        return a_raw, new_hidden_state

    def log_prob_t(self, state, hidden_state ,action):
        """
        Compute log probability of an action. 
        Here, action is assumed to be the transformed action.
        We use the stored raw sample (_last_a_raw) and adjust the log probability with the Jacobian of tanh.
        """
        #if self._last_dist is None or self._last_a_raw is None:
        #    raise ValueError("No stored distribution or raw action found. Make sure draw_action_t was called.")
        
        # Retrieve the raw log probability from the stored distribution
        #log_prob_raw = self._last_dist.log_prob(self._last_a_raw)
        
        # If the log_prob_raw is multi-dimensional (e.g., for multi-dimensional actions),
        # sum across the appropriate dimension (usually the last one).
        #if log_prob_raw.dim() > 1:
        #    log_prob_raw = log_prob_raw.sum(dim=-1)
        
        # Compute the Jacobian correction for the tanh transformation.
        # The derivative of tanh is: d/dx tanh(x) = 1 - tanh(x)**2.
        #eps = 1e-6  # for numerical stability
        #log_det_jacobian = torch.log(1.0 - torch.tanh(self._last_a_raw)**2 + eps)
        #if log_det_jacobian.dim() > 1:
        #    log_det_jacobian = log_det_jacobian.sum(dim=-1)
        
        # Adjust the log probability with the Jacobian correction.
        #log_prob = log_prob_raw - log_det_jacobian

        # Expand dimensions if needed (for consistent output shape, e.g., [batch_size, 1])
        dist, _ = self.distribution_t(state, hidden_state)
        return dist.log_prob(action)[:, None] #log_prob.unsqueeze(-1) if log_prob.dim() == 1 else log_prob

    def entropy_t(self, state=None):
        """
        Returns the entropy of the policy.
        """
        return self._action_dim / 2 * np.log(2 * np.pi * np.e) + torch.sum(self._log_sigma)

    def distribution_t(self, state, hidden_state):
        """
        Computes the distribution over actions given state and hidden state.
        
        Args:
            state (tensor): the current state.
            hidden_state (tensor or tuple): the current hidden state.
            
        Returns:
            A tuple (distribution, new_hidden_state) where:
                - distribution is a torch.distributions.MultivariateNormal instance.
                - new_hidden_state is the updated hidden state.
        """
        mu, chol_sigma, new_hidden_state = self.get_mean_and_chol(state, hidden_state)
        dist = torch.distributions.MultivariateNormal(loc=mu, scale_tril=chol_sigma, validate_args=False)
        self._last_dist = dist  # Save the distribution for later use.
        return dist, new_hidden_state

    def get_mean_and_chol(self, state, hidden_state):
        """
        Computes the mean and the Cholesky factor (diagonal matrix) of the covariance.
        Also returns the updated hidden state from the RNN regressor.
        
        Args:
            state (tensor): the current state.
            hidden_state (tensor or tuple): the current hidden state.
            
        Returns:
            A tuple (mu, chol_sigma, new_hidden_state) where:
                - mu: the mean computed by the regressor.
                - chol_sigma: the diagonal matrix computed from the exponentiation of log_sigma.
                - new_hidden_state: the updated hidden state.
        """
        # Ensure the standard deviations are positive.
        assert torch.all(torch.exp(self._log_sigma) > 0)
        
        # Assuming self._mu is now an RNN regressor that takes and returns hidden_state.
        mu, new_hidden_state = self._mu(state, hidden_state, **self._predict_params, output_tensor=True)
        chol_sigma = torch.diag(torch.exp(self._log_sigma))
        return mu, chol_sigma, new_hidden_state

    def set_weights(self, weights):
        log_sigma_data = torch.from_numpy(weights[-self._action_dim:])
        if self.use_cuda:
            log_sigma_data = log_sigma_data.cuda()
        self._log_sigma.data = log_sigma_data
        self._mu.set_weights(weights[:-self._action_dim])

    def get_weights(self):
        mu_weights = self._mu.get_weights()
        sigma_weights = self._log_sigma.data.detach().cpu().numpy()
        return np.concatenate([mu_weights, sigma_weights])

    def parameters(self):
        return chain(self._mu.model.network.parameters(), [self._log_sigma])


In [ ]:
#| export

class RL2PPO(Agent):
    """
    Proximal Policy Optimization (PPO) Agent supporting sequential data and RL² compatibility.
    """

    def __init__(self, mdp_info, policy, actor_optimizer, critic_params,
                 n_epochs_policy, meta_episodes_per_policy_update, meta_episodes_per_learner_batch, batch_size, eps_ppo, lam, ent_coeff=0.0,
                 critic_fit_params=None):
        """
        Constructor.

        Args:
            mdp_info (MDPInfo): Environment information.
            policy (TorchPolicy): Actor network.
            actor_optimizer (dict): Optimizer settings for actor.
            critic_params (dict): Parameters for critic network.
            n_epochs_policy (int or Parameter): PPO epochs per fit call.
            batch_size (int or Parameter): Batch size for PPO update.
            eps_ppo (float or Parameter): PPO clipping parameter.
            lam (float or Parameter): Lambda for GAE computation.
            ent_coeff (float or Parameter): Entropy regularization coefficient.
            critic_fit_params (dict): Optional parameters for fitting the critic.
        """
        # Save parameters
        self._critic_fit_params = dict(n_epochs=10) if critic_fit_params is None else critic_fit_params
        self._n_epochs_policy = to_parameter(n_epochs_policy)
        self.meta_episodes_per_policy_update = to_parameter(meta_episodes_per_policy_update)
        self.meta_episodes_per_learner_batch = to_parameter(meta_episodes_per_learner_batch)
        self._batch_size = to_parameter(batch_size)
        self._eps_ppo = to_parameter(eps_ppo)
        self._lambda = to_parameter(lam)
        self._ent_coeff = to_parameter(ent_coeff)
        self._gamma = to_parameter(mdp_info.gamma)
        # Build optimizer (for actor)
        self._optimizer = actor_optimizer['class'](policy.parameters(), **actor_optimizer['params'])

        # Build value function approximator (critic)
        self._V = Regressor(TorchApproximator, **critic_params)

        self.meta_episodes = []

        # Iteration counter
        self._iter = 1

        # Hidden states for actor and critic (for recurrent networks)
        self._actor_hidden_state = None
        self._critic_hidden_state = None

        # Save attributes for checkpointing
        self._add_save_attr(
            _critic_fit_params='pickle',
            _n_epochs_policy='mushroom',
            _batch_size='mushroom',
            _eps_ppo='mushroom',
            _lambda='mushroom',
            _ent_coeff='mushroom',
            _optimizer='torch',
            _V='mushroom',
            _iter='primitive'
        )

        super().__init__(mdp_info, policy, None)
        
        self.reset_hidden_state(1, self.policy.use_cuda)

    def reset_hidden_state(self, batch_size, device):
        """
        Reset the hidden states for both actor and critic networks.

        Args:
            batch_size (int): the batch size to initialize the hidden state.
            device (torch.device): the device on which to allocate the hidden state.
        """
        # Convert the device argument if it's a Boolean.
        device = torch.device('cuda') if device == "cuda" else torch.device('cpu')
        # Assumes that the actor network is accessible via self.policy.actor_network
        self._actor_hidden_state = self.policy._mu.model.network.init_hidden(
            batch_size=batch_size, device=device
        )
        # Assumes that the critic network is accessible via self._V.model.rnn.model
        self._critic_hidden_state = self._V.model.network.init_hidden(
            batch_size=batch_size, device=device
        )

    def draw_action(self, state, return_additional=True):
        """
        Overriding draw_action to handle hidden states and also save log-probabilities
        and value predictions during rollout.
        
        Args:
            state (numpy.ndarray): current state.
            
        Returns:
            action (tensor): the chosen action.
        """
        # Use the current actor hidden state with the policy's RL²-compatible draw_action_t.
        # This method returns both the sampled action and the updated hidden state.
        obs = np.expand_dims(state.astype(np.float32), axis=0)
        actor_hidden_state = self._actor_hidden_state
        action, new_actor_hidden_state = self.policy.draw_action_t(obs, actor_hidden_state)
        self._actor_hidden_state = new_actor_hidden_state

        # Evaluate critic with its hidden state:
        with torch.no_grad():
            # If your critic network is recurrent, pass the hidden state and update it.
            vpred, new_critic_hidden_state = self._V(obs, self._critic_hidden_state)
            self._critic_hidden_state = new_critic_hidden_state
            # Use the policy's stored distribution (computed in draw_action_t) for log-prob.
            logpac = self.policy.log_prob_t(obs, actor_hidden_state, action)
            action, logpac, vpred = action.squeeze(0), logpac.squeeze(0), vpred.squeeze(0)
        action = action.detach().cpu().numpy() if self.policy.use_cuda else action.detach().numpy()
        logpac = logpac.detach().cpu().numpy() if self.policy.use_cuda else logpac.detach().numpy()
        #vpred = vpred.detach().cpu().numpy() if self.policy.use_cuda else vpred.detach().numpy()
        if return_additional:
            return action, logpac, vpred
        return action
    
    def parse_meta_episode(self, meta_ep):
        states = []
        actions = []
        rewards = []
        next_states = []
        absorbing_flags = []
        logpacs = []
        vpreds = []
        for step in meta_ep:
            # Unpack each step (assuming it's a tuple as returned by _step)
            state, action, reward, next_state, absorbing, logpac, vpred, last = step
            states.append(state)
            actions.append(np.squeeze(action, 0))
            rewards.append(reward)
            next_states.append(next_state)
            absorbing_flags.append(absorbing)
            logpacs.append(np.squeeze(logpac, 0))
            vpreds.append(np.squeeze(vpred, 0))
        return {"obs":np.stack(states), "acs":np.stack(actions), "rews":np.array(rewards),
                "next_state":np.stack(next_states), "dones":np.array(absorbing_flags),
                "logpacs":np.stack(logpacs), "vpreds":np.stack(vpreds)}

    @torch.no_grad()
    def assign_credit(self, meta_episode: dict) -> dict:
        """
        Compute TD(lambda) returns and generalized advantage estimates for a meta-episode.
        
        In the meta-episodic setting of RL², the objective is to maximize the expected 
        discounted return of the meta-episode, so this function computes the advantages 
        and TD(lambda) returns over the entire meta-episode without applying the standard
        'done' masking.

        Args:
            meta_episode (dict): Meta-episode data structure with keys such as "rewards" and "vpreds".
            gamma (float): Discount factor.
            lam (float): GAE lambda decay parameter.

        Returns:
            dict: The input meta_episode extended with:
                - "advs": The generalized advantage estimates.
                - "tdlam_rets": The TD(lambda) returns.
        """
        # Assume that rewards and vpreds are NumPy arrays of shape (T,)
        T = meta_episode["rews"].shape[0]
        advs = np.zeros(meta_episode["vpreds"].shape, dtype=np.float32)
        
        for t in reversed(range(T)):
            r_t = meta_episode["rews"][t]
            V_t = meta_episode["vpreds"][t]
            # If not at the last timestep, use the next predicted value; else use 0.
            V_tp1 = meta_episode["vpreds"][t+1] if t+1 < T else 0.0
            # Retrieve advantage of next timestep if exists; else 0.
            A_tp1 = advs[t+1] if t+1 < T else 0.0
            
            # Compute temporal difference error.
            delta_t = r_t - V_t + self._gamma.get_value() * V_tp1
            # Compute the advantage using the recursive formula.
            advs[t][0] = delta_t + self._gamma.get_value() * self._lambda.get_value() * A_tp1

        # Store computed advantages and TD(lambda) returns back into the meta-episode dictionary.
        meta_episode["advs"] = advs
        meta_episode["tdlam_rets"] = meta_episode["vpreds"] + advs

        return meta_episode
        
    def fit(self, dataset, **info):
        """
        Fit the policy and value networks using PPO loss.

        Args:
            dataset (list): Dataset collected from environment.
            logpacs (Tensor or None): Log-probabilities of actions at rollout.
            vpreds (Tensor or None): Value predictions at rollout.
        """
        meta_ep = self.parse_meta_episode(dataset)
        meta_ep = self.assign_credit(meta_ep)
        self.meta_episodes.append(meta_ep)
        
        # 4. Check if we have collected enough meta-episodes to perform a policy update.
        if len(self.meta_episodes) >= self.meta_episodes_per_policy_update():
            # Update policy/actor and value network/critic for a number of PPO epochs.
            for opt_epoch in range(self._n_epochs_policy()):
                # Randomly shuffle indices of meta-episodes in our batch.
                idxs = np.random.permutation(self.meta_episodes_per_policy_update())
                # Process the meta-episode batch in minibatches.
                for i in range(0, self.meta_episodes_per_policy_update(), self.meta_episodes_per_learner_batch()):
                    mb_idxs = idxs[i:i + self.meta_episodes_per_learner_batch()]
                    mb_meta_eps = [self.meta_episodes[idx] for idx in mb_idxs]
                    # Compute losses using the list of meta-episodes in the minibatch.
                    losses = self.compute_losses(meta_episodes=mb_meta_eps)
                    # --- Update the policy network (actor) ---
                    self._optimizer.zero_grad()
                    losses['policy_loss'].backward()
                    self._optimizer.step()
                    # --- Update the value network (critic) ---
                    self._V.model._optimizer.zero_grad()
                    losses['value_loss'].backward()
                    self._V.model._optimizer.step()
            # After the update, clear the meta-episode buffer.
            self.meta_episodes = []
        self._iter += 1
    def compute_losses(self, meta_episodes: list) -> dict:
        """
        Computes the PPO losses (policy loss and value loss) for a batch of meta-episodes.
        
        Assumes each meta-episode is a dictionary with keys:
        "obs"         : Observations; shape (T, obs_dim)
        "acs"         : Actions; shape (T, action_dim)
        "rews"        : Rewards; shape (T,)
        "dones"       : Done flags; shape (T,)
        "logpacs"     : Old log probabilities; shape (T, 1) or (T,)
        "advs"        : Computed advantages; shape (T,)
        "tdlam_rets"  : TD(lambda) returns; shape (T,)
        
        The policy and value networks use the observations and a hidden state.
        
        Args:
            meta_episodes (list): List of meta-episode dictionaries.
            clip_param (float): PPO clipping parameter.
            ent_coef (float): Entropy bonus coefficient.
        
        Returns:
            dict: A dictionary with keys:
                "policy_loss", "value_loss", "meanent", "clipfrac" (all torch.Tensors).
        """
        # Helper to stack a given field from all meta-episodes.
        def get_tensor(field, dtype=None):
            # Each meta-episode already uses our names, so extract directly.
            mb_field = np.stack([meta_ep[field] for meta_ep in meta_episodes], axis=0)
            if dtype == "long":
                return torch.LongTensor(mb_field)
            return torch.FloatTensor(mb_field)
        
        # Convert stored fields to torch tensors.
        mb_obs     = get_tensor("obs")         # shape: (B, T, obs_dim)
        mb_acs     = get_tensor("acs", "long")   # shape: (B, T, action_dim)
        mb_rews    = get_tensor("rews")          # shape: (B, T)
        mb_dones   = get_tensor("dones")         # shape: (B, T)
        mb_logpacs = get_tensor("logpacs")       # shape: (B, T) or (B, T, 1)
        mb_advs    = get_tensor("advs")          # shape: (B, T)
        mb_tdlam_rets = get_tensor("tdlam_rets")  # shape: (B, T)
        
        B = len(meta_episodes)     # number of meta-episodes in the batch
        T = mb_obs.shape[1]        # time horizon per meta-episode

        # Get initial hidden states (here referred to as hidden_state) for the policy and value networks.
        hidden_state_policy = self.policy._mu.model.network.init_hidden(batch_size=B)
        hidden_state_value  = self._V.model.network.init_hidden(batch_size=B)

        # Forward pass for the policy.
        # We assume that our policy's distribution_t function accepts (obs, hidden_state)
        # and returns a tuple (distribution, new_hidden_state)
        pi_dists, _ = self.policy.distribution_t(mb_obs, hidden_state_policy)
        
        # Forward pass for the value network.
        # We assume that self._V's forward method accepts (obs, hidden_state) and returns (vpreds, new_hidden_state)
        vpreds, _ = self._V(mb_obs, hidden_state_value, output_tensor=True)
        
        # Compute extra quantities.
        entropies = self.policy.entropy_t()       # shape: (B, T) or (B, T, 1)
        logpacs_new = pi_dists.log_prob(mb_acs)  # shape: (B, T) or (B, T, 1)
        vpreds_new = vpreds
        
        # Compute the entropy bonus.
        meanent = torch.mean(entropies)
        policy_entropy_bonus = self._ent_coeff.get_value() * meanent
        
        # Compute the probability ratios (new log prob minus stored log prob)
        policy_ratios = torch.exp(logpacs_new - mb_logpacs)
        clipped_policy_ratios = torch.clamp(policy_ratios, 1 - self._eps_ppo.get_value(), 1 + self._eps_ppo.get_value())
        
        # Compute the surrogate loss terms.
        surr1 = mb_advs * policy_ratios
        surr2 = mb_advs * clipped_policy_ratios
        policy_surrogate_objective = torch.mean(torch.min(surr1, surr2))
        
        # Combined policy loss: negative surrogate objective minus entropy bonus.
        policy_loss = -(policy_surrogate_objective + policy_entropy_bonus)
        
        # Value loss computed via a Huber loss between TD(lambda) returns and the new value predictions.
        value_loss = torch.mean(self._V.model._loss(mb_tdlam_rets, vpreds_new))
        
        # Diagnostics: the clipping fraction (i.e., fraction of samples for which clipping occurred)
        clipfrac = torch.mean((surr1 > surr2).float())
        
        return {
            "policy_loss": policy_loss,
            "value_loss": value_loss,
            "meanent": meanent,
            "clipfrac": clipfrac
        }
        

    def _post_load(self):
        if self._optimizer is not None:
            update_optimizer_parameters(self._optimizer, list(self.policy.parameters()))


In [ ]:
#| export

class RL2PPOAgent(MushroomBaseAgent):
    """
    RL² PPO Agent for meta-learning, based on recurrent policy/value networks and MushroomRL core agent.
    """

    def __init__(self,
                 environment_info: MDPInfo,
                 hidden_layers_RNN: int = 1,
                 num_hidden_units_RNN: int = 64,
                 hidden_layers_MLP: List = None,
                 activation: str = "relu",
                 learning_rate_actor: float = 3e-4,
                 learning_rate_critic: float | None = None,
                 batch_size: int = 64,
                 n_epochs_policy: int = 4,
                 meta_episodes_per_policy_update: int = 1,
                 meta_episodes_per_learner_batch: int = 1,
                 eps_ppo: float = 0.2,
                 lam: float = 0.95,
                 ent_coeff: float = 0.0,
                 drop_prob: float = 0.0,
                 batch_norm: bool = False,
                 init_method: str = "xavier_uniform",
                 optimizer: str = "Adam",
                 loss: str = "MSE",
                 obsprocessors: list | None = None,
                 device: str = "cpu",
                 agent_name: str | None = "RL2PPO",
                 RNN_cell: str = "GRU",
                 std_0: float = 0.1, # Unused but kept for consistency
                 ):
        """
        Constructor. Sets up policy, critic, and PPO core agent.
        """

        # 1. Device setup
        self.n_steps_per_fit = None  # In RL² training loop, external control of fitting.
        use_cuda = self.set_device(device)

        # 2. Input shapes
        input_shape = self.get_input_shape(environment_info.observation_space)
        actor_output_shape = environment_info.action_space.shape
        input_shape = self.convert_recursively_to_int(input_shape)
        actor_output_shape = self.convert_recursively_to_int(actor_output_shape)

        # 3. Optimizers
        OptimizerClass = self.get_optimizer_class(optimizer)
        learning_rate_critic = learning_rate_critic or learning_rate_actor
        loss_function = self.get_loss_function(loss)

        # 4. Define actor network (RL² recurrent actor)
        hidden_layers_MLP = hidden_layers_MLP or [64, 64]
        
        policy_params = dict(
            network=RL2RNNActor,  # <== ⚡ RL² actor class
            input_shape=input_shape,
            output_shape=actor_output_shape,
            hidden_layers_RNN=hidden_layers_RNN,
            num_hidden_units_RNN=num_hidden_units_RNN,
            hidden_layers_MLP=hidden_layers_MLP,
            RNN_cell=RNN_cell,
            activation=activation,
            final_activation="identity",
            drop_prob=drop_prob,
            batch_norm=batch_norm,
            init_method=init_method,
            use_cuda=use_cuda,
            dropout=self.dropout,
        )

        policy = GaussianTorchPolicyRL2(**policy_params, mdp_info=environment_info)

        # 5. Define critic network (RL² recurrent value net)
        critic_params = dict(
            network=RL2RNNValue,  # <== ⚡ RL² critic class
            optimizer={'class': OptimizerClass, 'params': {'lr': learning_rate_critic}},
            loss=loss_function,
            input_shape=input_shape,
            output_shape=(1,),
            hidden_layers_RNN=hidden_layers_RNN,
            num_hidden_units_RNN=num_hidden_units_RNN,
            hidden_layers_MLP=hidden_layers_MLP,
            RNN_cell=RNN_cell,
            activation=activation,
            final_activation="identity",
            drop_prob=drop_prob,
            batch_norm=batch_norm,
            init_method=init_method,
            use_cuda=use_cuda,
            dropout=self.dropout,
        )

        actor_optimizer = {
            'class': OptimizerClass,
            'params': {'lr': learning_rate_actor}
        }

        # 6. Build the MushroomRL PPO core agent (with our RL² actor and critic)
        self.agent = RL2PPO(
            mdp_info=environment_info,
            policy=policy,
            actor_optimizer=actor_optimizer,
            critic_params=critic_params,
            n_epochs_policy=n_epochs_policy,
            meta_episodes_per_policy_update=meta_episodes_per_policy_update,
            meta_episodes_per_learner_batch=meta_episodes_per_learner_batch,
            batch_size=batch_size,
            eps_ppo=eps_ppo,
            lam=lam,
            ent_coeff=ent_coeff,
            critic_fit_params=None
        )

        # 7. Build the MushroomBaseAgent (super class)
        super().__init__(
            environment_info=environment_info,
            obsprocessors=obsprocessors,
            device=device,
            agent_name=agent_name
        )

        # 8. Logging networks
        logging.info("Actor (RL²) network:")
        if logging.getLogger().isEnabledFor(logging.INFO):
            input_size = self.add_batch_dimension_for_shape(input_shape)
            print(summary(self.actor, input_size=input_size))
            time.sleep(.2)

        logging.info("Critic (RL²) network:")
        if logging.getLogger().isEnabledFor(logging.INFO):
            input_size = self.add_batch_dimension_for_shape(input_shape)
            print(summary(self.critic, input_size=input_size))

    def reset_hidden(self, batch_size=1, device='cpu'):
        """
        Reset the hidden state of both policy (actor) and value (critic) networks.
        
        Args:
            batch_size (int): number of parallel episodes/tasks (usually 1).
            device (str): device where hidden states should be allocated ('cpu' or 'cuda').
        """
        self.agent.reset_hidden_state(batch_size=batch_size, device=device)
        # Reset actor hidden state
        #actor_network = self.agent.policy._mu._impl.model.network
        #actor_network.hidden_state = actor_network.model.rnn.model.init_hidden(batch_size=batch_size, device=device)

        # Reset critic hidden state
        #critic_network = self.agent._V._impl.model.network
        #critic_network.hidden_state = critic_network.model.rnn.model.init_hidden(batch_size=batch_size, device=device)

    def get_network_list(self, set_actor_critic_attributes: bool = True):
        """
        Returns the list of actor and critic networks for saving/loading.
        """
        critic = self.agent._V._impl.model.network
        actor = self.agent.policy._mu._impl.model.network

        networks = [critic, actor]

        if set_actor_critic_attributes:
            return networks, actor, critic
        else:
            return networks
